In [1]:
import os
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

In [3]:
llm.invoke("What is the capital of South Korea?").content

'Seoul.'

In [4]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'I don’t have real-time data access, so I can’t give you the exact current KRW→USD rate right now.\n\nWays to get the current rate quickly:\n- Google search: type “KRW to USD” or “KRW USD rate.”\n- Finance sites/apps: XE, OANDA, Bloomberg, Reuters, Yahoo Finance.\n- Your bank or card issuer’s rate (they’ll show the rate you’d actually get for a transfer or purchase).\n\nIf you want, I can help you convert an amount using a rate you provide. The formula is:\nUSD = KRW_amount / rate, where rate is the number of KRW per 1 USD.\n\nExample (illustrative only): at a rate of 1 USD = 1,350 KRW,\n2,500,000 KRW ≈ 2,500,000 / 1,350 ≈ 1,851.85 USD.\n\nTell me:\n- the amount in KRW you want to convert, and\n- the rate you want to use (or I can show the calculation with a placeholder rate).'}]


In [6]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [7]:
bfmsgs(result)

I don’t have real-time data access, so I can’t give you the exact current KRW→USD rate right now.

Ways to get the current rate quickly:
- Google search: type “KRW to USD” or “KRW USD rate.”
- Finance sites/apps: XE, OANDA, Bloomberg, Reuters, Yahoo Finance.
- Your bank or card issuer’s rate (they’ll show the rate you’d actually get for a transfer or purchase).

If you want, I can help you convert an amount using a rate you provide. The formula is:
USD = KRW_amount / rate, where rate is the number of KRW per 1 USD.

Example (illustrative only): at a rate of 1 USD = 1,350 KRW,
2,500,000 KRW ≈ 2,500,000 / 1,350 ≈ 1,851.85 USD.

Tell me:
- the amount in KRW you want to convert, and
- the rate you want to use (or I can show the calculation with a placeholder rate).

In [8]:
def get_fx(currency: str) -> str:
    """Get exchange rate for a given currency."""
    if currency.upper() in ["KRW", "USD"]:
        return "1500"

In [9]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [10]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
bfmsgs(result)

Based on the latest rate, 1 USD ≈ 1,500 KRW. So 1 KRW ≈ 0.0006667 USD.

Examples:
- 10,000 KRW ≈ 6.67 USD
- 1 KRW ≈ 0.0006667 USD

If you have a specific amount to convert, tell me and I’ll compute it. Rates can vary slightly by source, so let me know if you want a different reference rate.

In [11]:
# @tool # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from div id='wrapper'."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    text = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(text, "html.parser")
    wrapper = soup.find("div", id="wrapper")

    if wrapper:
        return wrapper.get_text(strip=True)
    return "Wrapper div not found"

In [12]:
fetch_text_from_url("https://www.hanyang.ac.kr/re12")

'LanguageThis provides an unofficial translation by the Google Translate. Information on International Admissions, Research, and Support is available on ourEnglish website검색#ABC#ABC#ABCENGHY-in기부MENUHY-in공지학부모기부사이트맵한양소개서울캠퍼스ERICA캠퍼스입학/교육연구/산학/창업한양 광장한양대는총장/경영상징과 UI설립/역사대학현황동문/발전기금한양대학교는건학 정신대학 규정 (학칙)Vision 2030윤리헌장조직도한양학원브랜드 아카이브대학평의원회인사말약력과 경력2026 중기발전계획총장 활동연설문역대 총장총장에게 바란다부총장/처장단한양의 상징교가UIUI 활용색상 및 서체한양 캐릭터슬로건설립자 메시지+연보한양의 역사최초/유일대학 역사관백서한눈에 보는 한양통계 연보대학 요람해외 자매결연 대학자체평가한양 지속가능경영한양 SDGs대학정보공시대외협력처 바로가기한양 동문발전기금 리포트발전기금 뉴스레터캠퍼스 소개학사 안내행정 지원 안내기관/단체대학 생활서울캠퍼스는대학/학과 소개대학원캠퍼스 지도+찾아오는길주차안내한양둘레길(8경)전화번호 안내학사일정수업 안내온라인 강좌전공제도학적변동성적졸업학점인정계절학기학점 교류업무별 담당 부서 안내등록 안내장학 안내증명발급 안내기숙사 안내교직이수교원자격증발급학생대관외부대관통합 양식자료실행정기관부속/부설기관학생 기구동아리 안내학생 언론사2025 캠퍼스 가이드북오늘의 메뉴IT 서비스복지 매장복지 혜택보건 안내의료혜택학생 상담인권센터학생증/국제학생증병무 안내고시반 안내캠퍼스 소개학사 안내행정 지원 안내기관/단체대학 생활ERICA캠퍼스는대학/학과 소개대학원연혁캠퍼스 지도+찾아오는길통학·셔틀버스주차안내ERICA 정부재정지원사업전화번호 안내학사일정수업 안내온라인 강좌전공제도학적변동성적졸업학점인정계절학기학점 교류업무별 담당 부서 안내등록 안내장학 안내기숙사 안내증명발급 안내교직 이수대관 안내통합 양식자

In [13]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [14]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT
)

In [15]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at Hanyang University? Find it at https://www.hanyang.ac.kr/re12"}]}
)
bfmsgs(result)

Here’s what the HYU “Today’s Menu” page shows for Seoul campus today (the page you linked includes Seoul and ERICA sections, but the excerpt you provided is the Seoul campus portion):

Seoul Campus – Today’s Menu (as of 2026-05-04)
- Breakfast – 천원의아침밥 (1000 won)
  - 파채돈육불고기 (Pork Bulgogi with green onion) with rice, tofu, kim, etc. 1,000원
  - 해물된장찌개 (Seafood Doenjang Stew) with rice, etc. 1,000원
  - 토스트/초코머핀 (Brunch Set) with salad, cereal & milk, eggs, etc. 1,000원
- Lunch
  - 떡갈비와사비마요덮밥 (Grilled Rib Patties with Wasabi Mayo on Rice) 4,500원
  - 나주곰탕 (Naju-style Ox Bone Soup) with rice and sides 6,500원
- Midday/Evening options (중식/석식) listed on the page include items like:
  - 김치어묵우동, 깐풍돈가스, 베이컨갈릭볶음밥, 우삼겹쫄면, 우삼겹마라로제떡볶이, 고추참치덮밥, 만두육개장
  - Note: Prices for these are not shown in the excerpt you provided.

Important notes
- Campus menus change daily. The page also has an “ERICA” (ERICA campus) today’s menu section, but the excerpt you shared shows only the Seoul campus details. If you’d like, I can fetch the ERICA today’s menu as well.

Would you like me to pull the ERICA campus today’s menu too, or check for any updates to Seoul campus today?